# Notebook 03 — Feature Engineering
Reads the real feature matrix produced by the preprocessing pipeline.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, pathlib
from scipy.signal import welch, coherence, butter, filtfilt
from scipy.integrate import trapezoid
from scipy import stats
import antropy as ant
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = pathlib.Path("data/preprocessed")
print(f"Feature file: {OUT/'all_subjects_features.csv'}")
df = pd.read_csv(OUT / "all_subjects_features.csv")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nLabel counts:")
print(df["label"].value_counts().rename({1:"pain", 0:"no-pain"}).to_string())
print(f"\nSubjects: {sorted(df['subject'].unique())}")

Feature file: data/preprocessed/all_subjects_features.csv
Shape: 781 rows × 140 columns

Label counts:
label
no-pain    414
pain       367

Subjects: ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005', 'sub-006', 'sub-007', 'sub-008', 'sub-009', 'sub-010', 'sub-011', 'sub-012', 'sub-013', 'sub-014', 'sub-015', 'sub-016', 'sub-017', 'sub-018', 'sub-019', 'sub-020', 'sub-021', 'sub-022', 'sub-023', 'sub-024', 'sub-025', 'sub-026']


In [2]:
# Feature group breakdown
meta = ["label","subject","gender","age","pain_threshold"]
feat_cols = [c for c in df.columns if c not in meta]

band_power_feats      = [f for f in feat_cols if any(b+"_" in f for b in ["delta","theta","alpha","beta","low_gamma"])
                         and not f.startswith(("coh","plv"))]
entropy_feats         = [f for f in feat_cols if any(k in f for k in ["samp_ent","perm_ent","spec_ent","higuchi","petrosian","dfa","lziv","hjorth"])]
time_domain_feats     = [f for f in feat_cols if any(k in f for k in ["rms","std","skew","kurt","ptp"])]
spectral_shape_feats  = [f for f in feat_cols if any(k in f for k in ["spec_cent","median_freq"])]
connectivity_feats    = [f for f in feat_cols if f.startswith(("coh_","plv_"))]

print(f"Total features:           {len(feat_cols)}")
print(f"  Band power (abs/rel/log+ratios): {len(band_power_feats)}")
print(f"  Nonlinear/entropy:               {len(entropy_feats)}")
print(f"  Time domain:                     {len(time_domain_feats)}")
print(f"  Spectral shape:                  {len(spectral_shape_feats)}")
print(f"  Connectivity (coh+PLV):          {len(connectivity_feats)}")

Total features:           135
  Band power (abs/rel/log+ratios): 51
  Nonlinear/entropy:               30
  Time domain:                     15
  Spectral shape:                  6
  Connectivity (coh+PLV):          30


In [3]:
# ANOVA-F feature ranking — real computation on the loaded data
from sklearn.feature_selection import f_classif, SelectKBest, VarianceThreshold

X = df[feat_cols].values.astype(np.float32)
y = df["label"].values.astype(int)

# Remove zero-variance features
var_sel = VarianceThreshold(1e-8)
X_var   = var_sel.fit_transform(X)
fc_var  = [feat_cols[i] for i in range(len(feat_cols)) if var_sel.get_support()[i]]
print(f"After variance filter: {X_var.shape[1]} / {len(feat_cols)} features kept")

# ANOVA-F on all remaining features
f_vals, p_vals = f_classif(X_var, y)

top_idx = np.argsort(f_vals)[::-1][:25]
print(f"\nTop 25 features by ANOVA F-score (pain vs no-pain):")
print(f"{'Rank':>4}  {'Feature':45s}  {'F-score':>10}  {'p-value':>12}")
print("-" * 78)
for rank, i in enumerate(top_idx, 1):
    print(f"{rank:4d}  {fc_var[i]:45s}  {f_vals[i]:10.2f}  {p_vals[i]:12.2e}")

After variance filter: 87 / 135 features kept

Top 25 features by ANOVA F-score (pain vs no-pain):
Rank  Feature                                           F-score       p-value
------------------------------------------------------------------------------
   1  CZ_dfa                                             108.22      8.02e-24
   2  CZ_higuchi                                         106.11      2.04e-23
   3  CZ_samp_ent                                         88.29      6.12e-20
   4  CZ_perm_ent                                         64.51      3.54e-15
   5  CZ_alpha_log                                        56.22      1.77e-13
   6  CZ_spec_ent                                         54.68      3.68e-13
   7  CZ_lziv                                             45.11      3.60e-11
   8  CZ_petrosian                                        41.14      2.45e-10
   9  C3_samp_ent                                         35.33      4.21e-09
  10  CZ_alpha_rel                        

In [4]:
# Mann-Whitney U tests — top 10 features
print(f"Mann-Whitney U tests (top 10 ANOVA features):")
print(f"{'Feature':45s}  {'U-statistic':>12}  {'p-value':>12}  {'Significant':>11}")
print("-" * 86)
for i in top_idx[:10]:
    fn   = fc_var[i]
    pv_  = df[df["label"]==1][fn].values
    nv_  = df[df["label"]==0][fn].values
    u, p = stats.mannwhitneyu(pv_, nv_, alternative="two-sided")
    sig  = "***" if p < 1e-10 else "**" if p < 0.001 else "*" if p < 0.05 else "n.s."
    print(f"{fn:45s}  {u:12.0f}  {p:12.2e}  {sig:>11}")

Mann-Whitney U tests (top 10 ANOVA features):
Feature                                         U-statistic       p-value  Significant
--------------------------------------------------------------------------------------
CZ_dfa                                               107729      5.92e-24          ***
CZ_higuchi                                            45919      1.30e-21          ***
CZ_samp_ent                                           47533      1.61e-19          ***
CZ_perm_ent                                           55274      4.80e-11          ***
CZ_alpha_log                                          99781      3.81e-14          ***
CZ_spec_ent                                           55341      5.55e-11          ***
CZ_lziv                                               57260      2.68e-09           **
CZ_petrosian                                          55614      9.83e-11          ***
C3_samp_ent                                           60236      5.73e-07           

In [5]:
# Pain vs no-pain mean comparison for top 8 features
print(f"Mean values — Pain vs No-Pain (top 8 features):")
print(f"{'Feature':40s}  {'Pain mean':>12}  {'No-Pain mean':>14}  {'Ratio':>8}")
print("-" * 80)
for i in top_idx[:8]:
    fn  = fc_var[i]
    pm  = df[df["label"]==1][fn].mean()
    nm  = df[df["label"]==0][fn].mean()
    rat = pm/nm if nm != 0 else float("nan")
    print(f"{fn:40s}  {pm:12.4f}  {nm:14.4f}  {rat:8.4f}")

Mean values — Pain vs No-Pain (top 8 features):
Feature                                      Pain mean    No-Pain mean     Ratio
--------------------------------------------------------------------------------
CZ_dfa                                          1.3982          1.2869    1.0865
CZ_higuchi                                      1.4989          1.5858    0.9452
CZ_samp_ent                                     0.7531          0.9634    0.7817
CZ_perm_ent                                     0.9026          0.9191    0.9820
CZ_alpha_log                                  -11.5363        -11.7481    0.9820
CZ_spec_ent                                     0.4809          0.5422    0.8868
CZ_lziv                                         0.3869          0.4587    0.8435
CZ_petrosian                                    1.0233          1.0247    0.9987


In [6]:
# Feature matrix info
npz = np.load("data/preprocessed/features_xy.npz", allow_pickle=True)
print("features_xy.npz contents:")
for k in npz.files:
    arr = npz[k]
    print(f"  {k:20s}: shape={arr.shape}  dtype={arr.dtype}")
print(f"\nFirst 5 feature names: {list(npz['feature_names'][:5])}")
print(f"Label distribution:    pain={( npz['y']==1).sum()}  no-pain={(npz['y']==0).sum()}")
print(f"Unique subjects:       {len(np.unique(npz['subjects']))}")

features_xy.npz contents:
  X                   : shape=(781, 135)  dtype=float32
  y                   : shape=(781,)  dtype=int64
  subjects            : shape=(781,)  dtype=object
  feature_names       : shape=(135,)  dtype=<U19

First 5 feature names: [np.str_('C3_delta_abs'), np.str_('C3_delta_rel'), np.str_('C3_delta_log'), np.str_('C3_theta_abs'), np.str_('C3_theta_rel')]
Label distribution:    pain=367  no-pain=414
Unique subjects:       26
